# 3. Inspect the built images

Checks artifact integrity and explains what consumes disk space. These checks do not authenticate the distribution channel.

**Tip:** start JupyterLab with `make lab`, which keeps execution outputs outside the committed source.

## Locate the repository

Jupyter kernels may start in `notebooks/` or `build/notebooks/`. Use the repository root for all relative paths.

In [ ]:
from pathlib import Path as _SCPath
import os as _sc_os
_start = _SCPath.cwd().resolve()
_candidates = (_start, *_start.parents)
_project_root = next(
    (path for path in _candidates if (path / "config/build.json").is_file() and (path / "notebooks").is_dir()),
    None,
)
if _project_root is None:
    raise RuntimeError("Open this notebook from inside the SC Linux checkout")
_sc_os.chdir(_project_root)
print("Repository:", _project_root)

In [ ]:
from pathlib import Path
import hashlib, json
output=Path.cwd().resolve()/'build/output'
artifacts=json.loads((output/'artifacts.json').read_text())
print(json.dumps(artifacts,indent=2))

## Recalculate SHA-256 digests

In [ ]:
for name,record in artifacts.items():
    file=output/name
    assert file.is_file(), f"Missing {file}"
    digest=hashlib.sha256()
    with file.open('rb') as stream:
        for chunk in iter(lambda:stream.read(1024*1024), b''):
            digest.update(chunk)
    assert (file.stat().st_size,digest.hexdigest())==(record['size'],record['sha256']), name
    print("OK",name)

## Review size measurements

In [ ]:
footprint=json.loads((output/'footprint.json').read_text())
print(json.dumps(footprint['artifacts'],indent=2))
print("Largest installed packages:")
for entry in footprint['packages']['largest'][:15]:
    print(f"{entry['installed_size_kib']:>9} KiB {entry['package']}")

The uncompressed RAW includes unused disk and filesystem metadata. The ISO includes the compressed RAW payload and installer files.